# Spatialization and Reverb

`locsig` places and moves sounds in the stereo (or n-channel) field; the reverb instruments add rooms. Together they finish a mix.

Docs: [CLM -- locsig and reverbs](https://ccrma.stanford.edu/software/snd/snd/sndclm.html)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import pysndlib.clm as clm
from pysndlib.instruments import fm_bell
from pysndlib.instruments import jc_reverb, nrev

SR = 44100

## locsig: static placement

`locsig(degree, distance, reverb)` places a source by angle: 0 degrees = full left channel, 90 = full right (for stereo); distance attenuates:

In [ ]:
with clm.Sound('placed.wav', channels=2, srate=SR):
    left = clm.make_locsig(0.0, 1.0, 0.0)     # hard left
    center = clm.make_locsig(45.0, 1.0, 0.0)  # center
    right = clm.make_locsig(90.0, 1.0, 0.0)   # hard right
    for n, loc in enumerate([left, center, right]):
        start = n * SR // 2
        osc = clm.make_oscil(440.0)
        amp = clm.make_env([0, 0, 10, 1, 90, 1, 100, 0], duration=0.4)
        for i in range(start, start + SR // 2):
            clm.locsig(loc, i - start, clm.env(amp) * clm.oscil(osc))

In [ ]:
placed, _ = clm.file2ndarray('placed.wav')
plt.figure(figsize=(14, 3))
plt.plot(placed[0], lw=0.6, label='L')
plt.plot(placed[1], lw=0.6, alpha=0.7, label='R')
plt.legend(); plt.title('three tones: left, center, right')
plt.show()

In [ ]:
ipd.Audio('placed.wav', rate=SR)

## Moving sources

`move_locsig(gen, degree, distance)` repositions a locsig at any time -- call it per sample for a trajectory across the room:

In [ ]:
with clm.Sound('moving.wav', channels=2, srate=SR):
    loc = clm.make_locsig(45.0, 1.0, 0.1)
    osc = clm.make_oscil(330.0)
    amp = clm.make_env([0, 0, 5, 1, 95, 1, 100, 0], duration=2.0)
    sweep = clm.make_triangle_wave(0.5, amplitude=45.0)   # 0.5 Hz: across the room and back
    for i in range(2 * SR):
        clm.move_locsig(loc, 45.0 + clm.triangle_wave(sweep), 1.0)
        clm.locsig(loc, i, clm.env(amp) * clm.oscil(osc))

In [ ]:
moving, _ = clm.file2ndarray('moving.wav')
plt.figure(figsize=(14, 3))
plt.plot(moving[0], lw=0.6, label='L')
plt.plot(moving[1], lw=0.6, alpha=0.7, label='R')
plt.legend(); plt.title('a tone gliding from left to right'); plt.show()

In [ ]:
ipd.Audio('moving.wav', rate=SR)

## Reverbs

The reverb instruments take the dry mix (via an intermediate revfile), diffuse it, and add it back. `jc_reverb` is the classic bright plate; `nrev` is the denser "new" reverb. Pass them to `Sound` with their parameters:

In [ ]:
with clm.Sound('nrev.wav', channels=2, srate=SR,
               reverb=nrev, reverb_data={'reverb_factor': 1.09, 'decay_time': 2.0},
               reverb_channels=2, revfile='nrev.rev'):
    fm_bell(0.0, 2.0, 440.0, 0.4)
    fm_bell(1.0, 2.0, 660.0, 0.3)

In [ ]:
ipd.Audio('nrev.wav', rate=SR)

In [ ]:
with clm.Sound('jcrev.wav', channels=2, srate=SR,
               reverb=jc_reverb, reverb_data={'decay_time': 3.0},
               reverb_channels=2, revfile='jcrev.rev'):
    fm_bell(0.0, 2.0, 440.0, 0.4)

In [ ]:
ipd.Audio('jcrev.wav', rate=SR)

## Compare decay times

In [ ]:
for dt in (0.5, 1.0, 3.0):
    name = f'rev_{dt}.wav'
    with clm.Sound(name, channels=1, srate=SR,
                   reverb=jc_reverb, reverb_data={'decay_time': dt},
                   reverb_channels=1, revfile=f'rev_{dt}.rev'):
        fm_bell(0.0, 1.0, 440.0, 0.4)
    d, _ = clm.file2ndarray(name)
    plt.plot(d[0], lw=0.5, label=f'decay_time={dt}')
plt.legend(); plt.title('reverb tails: same dry sound, different rooms')
plt.show()

## Summary

- `locsig` places; `move_locsig` pans; degree/distance/reverb are the knobs
- Reverb instruments process an intermediate file -- `Sound(..., reverb=...)` wires it

That is the end of the series. Back to [the first notebook](01_Basic.ipynb) or the [library docs](https://testcase.github.io/pysndlib/index.html).